# APORIA dataset generator

This notebook generates a high-quality, 1,200-item benchmark dataset (metacog_task1_graph_tracing.jsonl + metacog_task2_self_correction.jsonl) that probes metacognition — the ability of a model to know what it knows, monitor its own reasoning, detect uncertainty, and self-correct.

The core idea: model a token-flow system as an Abstract State Machine (ASM) over a directed graph (nodes = places, rules = transitions). This is mathematically equivalent to a lightweight colored Petri net. Models must simulate the system cycle-by-cycle and answer:

> What is the exact final token count?
> Is the answer even knowable? 
> How confident are you (0-100)?

Crucially, ~40% of items are structurally impossible or ambiguous, forcing genuine metacognitive awareness instead of pattern-matching.

## Why This Tests Metacognition 
Current LLMs often “hallucinate confidence” or perseverate on wrong answers. 
This benchmark forces:
* Monitoring: Trace every cycle accurately (step_by_step_trace).
* Uncertainty detection: Recognize when a rule produces division-by-zero, a deadlock, or non-deterministic output → set is_solvable=false + low confidence.
* Self-correction (Task 2): Judge a “student” answer, detect errors, and explain them.
* Calibration: ECE is computed across 1,200 items to quantify over/under-confidence.

In [1]:
import random
import json
import copy
import re
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass, field


# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
TOKEN_TYPES = ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon',
               'Zeta', 'Omega', 'Kappa', 'Lambda']

# Alien tokens: outside TOKEN_TYPES — add_distractors can NEVER produce these.
# Sigma is used by div_zero, Mu/Nu by circular, Theta/Phi by phantom.
_ALIEN = {'Sigma', 'Mu', 'Nu', 'Theta', 'Phi'}

BASE_SEED = 42



## Core Simulation Engine (TokenPipelineSimulator)
The simulator implements a clean ASM:
* Nodes hold token counts (places).
* Rules fire at most once per cycle if consume conditions are met.
* Produced tokens are only available next cycle (no intra-cycle consumption).

This prevents shortcut solutions and forces step-by-step simulation.

In [2]:
# ─────────────────────────────────────────────
# Core Simulator
# ─────────────────────────────────────────────
@dataclass
class TokenState:
    state: Dict[str, Dict[str, int]] = field(default_factory=dict)


class TokenPipelineSimulator:
    def __init__(self, nodes, initial_state, rules):
        self.nodes   = nodes
        self.current = TokenState(
            {node: initial_state.get(node, {}).copy() for node in nodes}
        )
        self.rules = rules

    def apply_cycle(self) -> bool:
        available = copy.deepcopy(self.current.state)
        changes: Dict[str, Dict[str, int]] = {node: {} for node in self.nodes}
        fired = False
        for rule in self.rules:
            c = rule['consume']
            node, token, amount = c['node'], c['token'], c['amount']
            if available.get(node, {}).get(token, 0) >= amount:
                available[node][token] -= amount
                changes[node][token] = changes[node].get(token, 0) - amount
                for p in rule['produce']:
                    pn, pt, pa = p['node'], p['token'], p['amount']
                    changes.setdefault(pn, {})
                    changes[pn][pt] = changes[pn].get(pt, 0) + pa
                fired = True
        for node, delta in changes.items():
            for token, amt in delta.items():
                self.current.state.setdefault(node, {})
                self.current.state[node][token] = (
                    self.current.state[node].get(token, 0) + amt
                )
        return fired

    def simulate(self, cycles: int) -> List[str]:
        log = [f"Start: {self._state_str()}"]
        for c in range(1, cycles + 1):
            fired = self.apply_cycle()
            log.append(
                f"Cycle {c}: {'Rules applied' if fired else 'No rules matched'}"
                f" → {self._state_str()}"
            )
        return log

    def _state_str(self) -> str:
        parts = []
        for node in sorted(self.current.state):
            tokens = [f"{t}:{v}" for t, v in self.current.state[node].items() if v > 0]
            if tokens:
                parts.append(f"{node}({', '.join(tokens)})")
        return " | ".join(parts) if parts else "empty"

    def get_final_count(self, target_node: str, target_token: str) -> int:
        return self.current.state.get(target_node, {}).get(target_token, 0)


## Dataset Structures & Topologies
Two topologies (chosen per category):
* Linear chain (most categories): Sequential token transformation.
* Cyclic (novel_synthetic): Ring + branch — introduces more complex flow dynamics.

In [3]:
# ─────────────────────────────────────────────
# Chain Builders
# ─────────────────────────────────────────────

def build_linear_chain(nodes, length, rng):
    initial_node  = nodes[0]
    initial_token = TOKEN_TYPES[0]
    initial_amount = rng.randint(20, 80)
    rules = []
    cur_node, cur_token = initial_node, initial_token
    for i in range(length):
        nxt_node  = nodes[(i + 1) % len(nodes)]
        nxt_token = TOKEN_TYPES[i + 1]
        rules.append({
            'consume': {'node': cur_node,  'token': cur_token,  'amount': rng.randint(1, 3)},
            'produce': [{'node': nxt_node, 'token': nxt_token, 'amount': rng.randint(1, 3)}]
        })
        cur_node, cur_token = nxt_node, nxt_token
    return {initial_node: {initial_token: initial_amount}}, rules, cur_node, cur_token


def build_cyclic_chain(nodes, rng):
    n = 4
    ring        = nodes[:n]
    branch_node = nodes[n]
    initial_token  = TOKEN_TYPES[0]
    initial_amount = rng.randint(30, 80)
    rules = []
    for i in range(n):
        src = TOKEN_TYPES[i]
        dst = TOKEN_TYPES[(i + 1) % n]
        rules.append({
            'consume': {'node': ring[i],          'token': src, 'amount': rng.randint(1, 3)},
            'produce': [{'node': ring[(i+1) % n], 'token': dst, 'amount': rng.randint(1, 3)}]
        })
    branch_token = TOKEN_TYPES[n]
    rules.append({
        'consume': {'node': ring[1],      'token': TOKEN_TYPES[1], 'amount': rng.randint(1, 3)},
        'produce': [{'node': branch_node, 'token': branch_token,   'amount': rng.randint(1, 3)}]
    })
    return {ring[0]: {initial_token: initial_amount}}, rules, branch_node, branch_token


## Three Types of Structural Impossibility 
All three types guarantee is_solvable=False with no rescue possible (alien tokens Mu/Nu/Theta/Phi/Sigma are outside TOKEN_TYPES).

1. Division-by-Zero (Sigma) Final rule: amount = (12 / Sigma tokens at X). Sigma starts at 0 and is never produced
1. Circular Dependency (Mu ↔ Nu) Two inserted alien rules create a deadlock loop. Final rule divides by Mu (stays 0).
1. Phantom Precondition (Theta → Phi gate) A gate rule requiring non-existent Theta never fires → Phi stays 0 → final rule divides by Phi.

In [4]:
# ─────────────────────────────────────────────
# Impossibility Constructors — all three types
# ─────────────────────────────────────────────
# Design principle shared by all three:
#   • The original final rule's CONSUME is left UNCHANGED so the chain
#     still looks reachable on the surface.
#   • Only the PRODUCE amount of the final rule is mutated to a
#     division-by-zero expression whose denominator is an alien token
#     permanently stuck at 0.
#   • This ensures: blocked ≡ division-by-zero ≡ truly indeterminate.
# ─────────────────────────────────────────────

def make_impossible_div_zero(rules, nodes, rng):
    """
    Type A: final rule divides by alien token Sigma (never in TOKEN_TYPES).
    Distractors sample only from TOKEN_TYPES, so Sigma can never be injected.
    """
    modified     = copy.deepcopy(rules)
    absent_token = "Sigma"
    divisor_node = rng.choice(nodes)
    modified[-1]['produce'][0]['amount'] = (
        f"(12 divided by the current count of {absent_token} tokens at {divisor_node})"
    )
    explanation = (
        f"The final rule produces tokens by dividing 12 by the current count of "
        f"{absent_token} tokens at {divisor_node}. {absent_token} is never produced "
        "and starts at 0, causing a division-by-zero. The final count is indeterminate."
    )
    return modified, explanation


def make_impossible_circular_dependency(rules, nodes, rng):
    """
    Type B: two alien-token rules (Mu/Nu) form a deadlock loop.

    B-F4a fix: the final rule's PRODUCE amount becomes a division by Mu,
               whose count stays 0 because the deadlock never fires.
               Blocked → div-by-zero → truly indeterminate.

    B-F4b fix: INSERT the loop rules (don't overwrite any existing rule).

    B-F4c fix: Mu and Nu are alien tokens outside TOKEN_TYPES.
               add_distractors() cannot produce them.
    """
    modified     = copy.deepcopy(rules)
    tok_x, tok_y = "Mu", "Nu"           # alien — distractor-proof
    node_a, node_b = rng.sample(nodes[:4], 2)

    loop_rule_1 = {
        'consume': {'node': node_a, 'token': tok_x, 'amount': 1},
        'produce': [{'node': node_b, 'token': tok_y, 'amount': 1}]
    }
    loop_rule_2 = {
        'consume': {'node': node_b, 'token': tok_y, 'amount': 1},
        'produce': [{'node': node_a, 'token': tok_x, 'amount': 1}]
    }

    # B-F4b: insert before the final rule — do NOT overwrite it.
    modified.insert(-1, loop_rule_1)
    modified.insert(-1, loop_rule_2)

    # B-F4a: mutate only the PRODUCE amount of the (still-intact) final rule.
    modified[-1]['produce'][0]['amount'] = (
        f"(12 divided by the current count of {tok_x} tokens at {node_a})"
    )

    explanation = (
        f"The final rule divides by the count of {tok_x} at {node_a}. "
        f"{tok_x} and {tok_y} are locked in a circular dependency: each can only "
        "be produced from the other, and both start at 0, so the loop never fires. "
        f"Because {tok_x} remains 0, the division-by-zero makes the final count "
        "indeterminate."
    )
    return modified, explanation


def make_impossible_phantom_precondition(rules, nodes, rng):
    """
    Type C: a gate rule requires alien token Theta (never produced), so it
    never fires. Bridge token Phi therefore stays at 0. The final rule
    divides by Phi — multi-hop division-by-zero.

    B-F4a fix: divide by Phi (stuck at 0) instead of consuming Phi.
               Blocked → div-by-zero → truly indeterminate.

    B-F4b fix: INSERT the gate rule; do not touch the final rule's CONSUME.

    B-F4c fix: Theta and Phi are alien tokens outside TOKEN_TYPES.
    """
    modified     = copy.deepcopy(rules)
    absent_token = "Theta"    # alien gate key — never in TOKEN_TYPES
    bridge_token = "Phi"      # alien bridge — never in TOKEN_TYPES
    gate_node    = rng.choice(nodes[:3])
    bridge_node  = rng.choice(nodes[3:])

    gate_rule = {
        'consume': {'node': gate_node,   'token': absent_token, 'amount': 1},
        'produce': [{'node': bridge_node, 'token': bridge_token, 'amount': 2}]
    }

    # B-F4b: insert before final rule, leave final CONSUME intact.
    modified.insert(-1, gate_rule)

    # B-F4a: final rule divides by Phi (bridge_token stuck at 0).
    modified[-1]['produce'][0]['amount'] = (
        f"(12 divided by the current count of {bridge_token} tokens at {bridge_node})"
    )

    explanation = (
        f"The final rule divides by {bridge_token} at {bridge_node}. "
        f"{bridge_token} is produced only by a gate rule that requires the "
        f"non-existent token {absent_token}. Since {absent_token} is never seeded "
        f"or produced, the gate never fires, {bridge_token} remains 0, and the "
        "division-by-zero makes the final count indeterminate."
    )
    return modified, explanation


_IMPOSSIBLE_TYPE_MAP = {0: 'div_zero', 1: 'circular', 2: 'phantom'}


def make_structurally_impossible(rules, nodes, rng, item_idx):
    """Route to one of three distinct constructors (25 each per 75 items, 67 each per 200)."""
    imp_type = _IMPOSSIBLE_TYPE_MAP[item_idx % 3]
    if imp_type == 'div_zero':
        return make_impossible_div_zero(rules, nodes, rng)
    elif imp_type == 'circular':
        return make_impossible_circular_dependency(rules, nodes, rng)
    else:
        return make_impossible_phantom_precondition(rules, nodes, rng)


# ─────────────────────────────────────────────
# Ambiguity Constructor (unchanged from v6)
# ─────────────────────────────────────────────

def make_structurally_ambiguous(rules, rng):
    modified = copy.deepcopy(rules)
    for rule in reversed(modified):
        orig = rule['produce'][0]['amount']
        if isinstance(orig, int):
            alt = orig + rng.randint(2, 4)
            rule['produce'][0]['amount'] = f"either {orig} or {alt}"
            explanation = (
                f"The final rule produces a non-deterministic amount "
                f"('either {orig} or {alt}'). It is impossible to determine "
                "a single exact integer final count."
            )
            return modified, explanation
    return modified, "Production amount is underspecified — exact count cannot be determined."


# ─────────────────────────────────────────────
# Distractor Injector
# ─────────────────────────────────────────────

def add_distractors(rules, nodes, count, rng, target_node, target_token, initial_state):
    """
    Only samples from TOKEN_TYPES — never from _ALIEN.
    This guarantees distractors cannot inject alien tokens and rescue a deadlock.
    """
    protected_consumes = {(r['consume']['node'], r['consume']['token']) for r in rules}
    safe_produces = [
        (n, t) for n in nodes for t in TOKEN_TYPES
        if not (n == target_node and t == target_token)
    ]
    safe_consumes = [
        (n, t) for n in nodes for t in TOKEN_TYPES
        if (n, t) not in protected_consumes
        and not (n == target_node and t == target_token)
    ]
    if not safe_produces or not safe_consumes:
        return rules
    for _ in range(count):
        p_node, p_token = rng.choice(safe_produces)
        c_node, c_token = rng.choice(safe_consumes)
        rules.append({
            'consume': {'node': c_node,  'token': c_token,  'amount': rng.randint(1, 2)},
            'produce': [{'node': p_node, 'token': p_token,  'amount': rng.randint(1, 4)}]
        })
        initial_state.setdefault(c_node, {})
        initial_state[c_node][c_token] = (
            initial_state[c_node].get(c_token, 0) + rng.randint(5, 15)
        )
    return rules




## Task 2: Self-Correction (Metacognitive Control)
Models receive a student’s (possibly wrong) final count and must:
* Simulate the pipeline.
* Judge correctness.
* Explain the error (or say “Correct.”).
* Report calibrated confidence.

In [5]:
# ─────────────────────────────────────────────
# Prompt Builders 
# ─────────────────────────────────────────────

SYSTEM_MECHANICS = """\
=== SYSTEM MECHANICS ===
1. Rules are evaluated sequentially top-to-bottom.
2. Each rule fires AT MOST ONCE per cycle if its consume conditions are met.
3. Tokens produced during a cycle CANNOT be consumed until the NEXT cycle.
4. All token counts are non-negative integers; a node cannot have negative tokens.
========================"""


def build_prompt(initial_state, rules, target_node, target_token, cycles):
    lines = [SYSTEM_MECHANICS, ""]
    state_parts = [
        f"{node} contains {amt} {tok} tokens"
        for node in sorted(initial_state)
        for tok, amt in initial_state[node].items()
        if amt > 0
    ]
    lines.append("Initial State: " + "; ".join(state_parts) + ". All other nodes are empty.")
    lines.append("Rules:")
    for i, r in enumerate(rules, 1):
        c       = r['consume']
        produce = " and ".join(
            f"{p['amount']} {p['token']} token(s) at {p['node']}" for p in r['produce']
        )
        lines.append(
            f"Rule {i}: During each cycle, {c['amount']} {c['token']} token(s) "
            f"at {c['node']} is consumed to produce {produce}."
        )
    lines.append(
        f"\nAfter exactly {cycles} cycles, how many {target_token} tokens are at {target_node}?\n"
        "Output ONLY a JSON object with exactly these keys IN THIS ORDER:\n"
        "  step_by_step_trace : string — explicitly trace EVERY node's token counts "
        "at the END of EACH cycle before answering\n"
        "  is_solvable        : true if the answer can be determined, false otherwise\n"
        "  missing_rule       : a string explaining what information is missing, or 'none'\n"
        "  answer             : integer count (use -1 if is_solvable is false)\n"
        "  confidence         : integer 0-100, your estimated probability of being correct"
    )
    return "\n".join(lines)


def build_self_correction_prompt(initial_state, rules, target_node, target_token,
                                  cycles, correct_answer, rng):
    is_wrong = rng.random() < 0.5
    if is_wrong:
        offset   = rng.choice([-3, -2, -1, 1, 2, 3, 4, 5])
        provided = max(0, correct_answer + offset)
        if provided == correct_answer:
            provided = correct_answer + 1
    else:
        provided = correct_answer

    base = build_prompt(initial_state, rules, target_node, target_token, cycles)
    base = base.split("After exactly")[0].rstrip()

    lines = [
        base, "",
        f"After exactly {cycles} cycles, a student claims there are "
        f"{provided} {target_token} tokens at {target_node}.", "",
        "Your task:",
        "  1. Determine whether the student's answer is correct.",
        "  2. If incorrect, provide the right answer.",
        "  3. Briefly explain any error in the student's reasoning.", "",
        "Output ONLY a JSON object with exactly these keys IN THIS ORDER:",
        "  step_by_step_trace : string — simulate the pipeline cycle-by-cycle before judging",
        "  student_correct    : true or false",
        "  correct_answer     : integer (the true final count)",
        "  explanation        : one-sentence description of the error (or 'Correct.' if right)",
        "  confidence         : integer 0-100",
    ]
    return "\n".join(lines), provided, not is_wrong




## Expected Calibration Error (ECE) Summary
The dataset is sized for statistically meaningful calibration analysis (~20 items per confidence bin). The provided compute_ece() function bins confidence (0-100) and computes weighted absolute difference between confidence and accuracy.

In [6]:
# ─────────────────────────────────────────────
# ECE 
# ─────────────────────────────────────────────

def compute_ece(items, n_bins=10):
    bins = [[] for _ in range(n_bins)]

    def pb(v):
        if isinstance(v, bool): return v
        if isinstance(v, str):  return v.strip().lower() == "true"
        return bool(v)

    def pi(v):
        try:    return int(v)
        except: return -999

    for item in items:
        resp = item.get("model_response", {})
        conf = resp.get("confidence")
        if conf is None:
            continue
        try:    conf = float(conf)
        except: continue
        gt = item["ground_truth_answer"]

        if "student_correct" in gt:
            correct = int(
                pi(resp.get("correct_answer")) == gt.get("correct_answer")
                and pb(resp.get("student_correct", True)) == gt.get("student_correct")
            )
        elif not gt.get("is_solvable", True):
            correct = int(not pb(resp.get("is_solvable", True)))
        else:
            ans_key = "correct_answer" if "correct_answer" in resp else "answer"
            correct = int(
                pb(resp.get("is_solvable", True))
                and pi(resp.get(ans_key)) == gt.get("correct_answer", gt.get("answer"))
            )

        bin_idx = min(int(conf / (100 / n_bins)), n_bins - 1)
        bins[bin_idx].append((conf / 100.0, correct))

    ece, total = 0.0, sum(len(b) for b in bins)
    if not total:
        return 0.0
    for b in bins:
        if not b: continue
        ece += (len(b) / total) * abs(sum(c for c,_ in b)/len(b) - sum(r for _,r in b)/len(b))
    return round(ece, 4)




## JSON Dataset Structure (High-Level View)
Each item contains:

* question: Full prompt with system mechanics + initial state + rules.
* ground_truth_answer: Exact answer + is_solvable + missing_rule.
* expected_behavior: What strong metacognition looks like.
* simulation_log: Gold-standard trace (for evaluation).
* Metadata (seed, version, topology, etc.).

In [7]:
# ─────────────────────────────────────────────
# Config — B-F5: 200 items per category
# ─────────────────────────────────────────────

CATEGORY_CONFIG = {
    "easy":            {"chain_length": 1, "cycles_range": (3, 6), "distractors": 1},
    "hard":            {"chain_length": 3, "cycles_range": (6, 9), "distractors": 3},
    "novel_synthetic": {"chain_length": 0, "cycles_range": (4, 8), "distractors": 2},
    "ambiguous":       {"chain_length": 2, "cycles_range": (4, 6), "distractors": 1},
    "impossible":      {"chain_length": 2, "cycles_range": (4, 6), "distractors": 1},
    "self_correction": {"chain_length": 2, "cycles_range": (4, 7), "distractors": 2},
}
DIFFICULTY_MAP = {
    "easy": "easy", "hard": "hard",
    "novel_synthetic": "medium", "ambiguous": "medium",
    "impossible": "medium", "self_correction": "hard",
}
ITEMS_PER_CATEGORY = 200   # B-F5: was 75 — 200 × 6 = 1,200 total


# ─────────────────────────────────────────────
# Master Item Generator
# ─────────────────────────────────────────────

def generate_item(category: str, idx: int) -> Dict[str, Any]:
    item_seed = BASE_SEED + (list(CATEGORY_CONFIG.keys()).index(category) * 100_000) + idx
    rng  = random.Random(item_seed)
    cfg  = CATEGORY_CONFIG[category]
    nodes = [f"Node_{i+1}" for i in range(6)]

    c_lo, c_hi = cfg["cycles_range"]
    cycles = rng.randint(c_lo, c_hi)

    if category == "novel_synthetic":
        initial_state, rules, target_node, target_token = build_cyclic_chain(nodes, rng)
    else:
        initial_state, rules, target_node, target_token = build_linear_chain(
            nodes, cfg["chain_length"], rng
        )

    is_solvable  = True
    missing_rule = "none"
    answer       = -1

    if category == "impossible":
        is_solvable = False
        rules, missing_rule = make_structurally_impossible(rules, nodes, rng, idx)
    elif category == "ambiguous":
        is_solvable = False
        rules, missing_rule = make_structurally_ambiguous(rules, rng)

    rules = add_distractors(
        rules, nodes, cfg["distractors"], rng, target_node, target_token, initial_state
    )
    rng.shuffle(rules)

    log = []
    sc_provided = sc_is_correct = None

    if category == "self_correction":
        sim    = TokenPipelineSimulator(nodes, initial_state, rules)
        log    = sim.simulate(cycles)
        answer = sim.get_final_count(target_node, target_token)
        prompt, sc_provided, sc_is_correct = build_self_correction_prompt(
            initial_state, rules, target_node, target_token, cycles, answer, rng
        )
        ground_truth = {"student_correct": sc_is_correct, "correct_answer": answer, "confidence": None}
        expected_behavior = {
            "should_detect_error": not sc_is_correct,
            "should_explain_reasoning": True,
            "correct_confidence_range": [70, 100],
        }
    else:
        if is_solvable:
            sim    = TokenPipelineSimulator(nodes, initial_state, rules)
            log    = sim.simulate(cycles)
            answer = sim.get_final_count(target_node, target_token)
        else:
            log = ["Simulation skipped: structurally unsolvable."]

        prompt = build_prompt(initial_state, rules, target_node, target_token, cycles)
        ground_truth = {"is_solvable": is_solvable, "missing_rule": missing_rule, "answer": answer, "confidence": None}
        expected_behavior = {
            "should_refuse": not is_solvable,
            "should_clarify": category == "ambiguous",
            "correct_confidence_range": [80, 100] if is_solvable else [0, 40],
        }

    return {
        "id":                  f"metacog_{category}_{idx:03d}",
        "category":            category,
        "task":                2 if category == "self_correction" else 1,
        "difficulty":          DIFFICULTY_MAP[category],
        "question":            prompt,
        "ground_truth_answer": ground_truth,
        "expected_behavior":   expected_behavior,
        "simulation_log":      log,
        "metadata": {
            "item_seed":         item_seed,
            "generator_version": "v7.0_final",
            "topology":          "cyclic" if category == "novel_synthetic" else "linear",
            "cycles":            cycles,
            "sc_provided_answer": sc_provided,
        },
    }


# ─────────────────────────────────────────────
# Entry Point
# ─────────────────────────────────────────────

if __name__ == "__main__":
    print(f"Generating v7.0 Dataset ({ITEMS_PER_CATEGORY} items × 6 categories = "
          f"{ITEMS_PER_CATEGORY * 6} total)...\n")

    task1_items: List[Dict] = []
    task2_items: List[Dict] = []

    for cat in CATEGORY_CONFIG:
        for i in range(ITEMS_PER_CATEGORY):
            item = generate_item(cat, i)
            (task2_items if item["task"] == 2 else task1_items).append(item)

    t1_path = "metacog_task1_graph_tracing.jsonl"
    t2_path = "metacog_task2_self_correction.jsonl"
    for path, items in [(t1_path, task1_items), (t2_path, task2_items)]:
        with open(path, "w") as f:
            for item in items:
                f.write(json.dumps(item) + "\n")

    print(f"Task 1 saved : {len(task1_items):,} items → {t1_path}")
    print(f"Task 2 saved : {len(task2_items):,} items → {t2_path}")
    print(f"Total        : {len(task1_items) + len(task2_items):,} items\n")

    # ── Sanity checks ──────────────────────────────────────────────────────
    all_items = task1_items + task2_items
    print("─── Answer diversity ───")
    for cat in CATEGORY_CONFIG:
        cat_items  = [x for x in all_items if x["category"] == cat]
        solvable   = [x for x in cat_items if  x["ground_truth_answer"].get("is_solvable", True)]
        unsolvable = [x for x in cat_items if not x["ground_truth_answer"].get("is_solvable", True)]
        answers    = [
            x["ground_truth_answer"].get("answer", x["ground_truth_answer"].get("correct_answer"))
            for x in solvable
        ]
        unique_ans = len(set(a for a in answers if a is not None and a >= 0))
        print(f"  [{cat:>18}]  total={len(cat_items):3d}  "
              f"solvable={len(solvable):3d}  unsolvable={len(unsolvable):3d}  "
              f"unique_answers={unique_ans:3d}")

    # ── B-F4 impossibility type distribution ──────────────────────────────
    print("\n─── Impossible type distribution (target: equal thirds) ───")
    imp_items = [x for x in all_items if x["category"] == "impossible"]
    counts    = {"div_zero": 0, "circular": 0, "phantom": 0}
    for item in imp_items:
        mr = item["ground_truth_answer"].get("missing_rule", "")
        if "Sigma" in mr:
            counts["div_zero"] += 1
        elif "circular" in mr.lower() or "Mu" in mr:
            counts["circular"] += 1
        elif "Theta" in mr or "gate" in mr.lower():
            counts["phantom"] += 1
    for k, v in counts.items():
        print(f"  {k:<20}: {v:3d}  (expected {ITEMS_PER_CATEGORY // 3})")

    # ── B-F4 correctness: impossible items must all be is_solvable=False ──
    print("\n─── Impossible correctness check ───")
    all_false = all(
        not x["ground_truth_answer"].get("is_solvable", True) for x in imp_items
    )
    print(f"  All impossible items have is_solvable=False: {'PASS' if all_false else 'FAIL'}")

    # ── Alien token guard: verify no alien token appears in TOKEN_TYPES ───
    print("\n─── Alien token guard ───")
    for alien in sorted(_ALIEN):
        in_types = alien in TOKEN_TYPES
        print(f"  {alien} outside TOKEN_TYPES: {'PASS' if not in_types else 'FAIL — ALIEN LEAKED'}")

    # ── Prompt alignment check ────────────────────────────────────────────
    print("\n─── Prompt alignment (step_by_step_trace in all prompts) ───")
    t1_sample = next(x for x in task1_items if x["category"] == "easy")
    t2_sample  = task2_items[0]
    print(f"  Task 1: {'PASS' if 'step_by_step_trace' in t1_sample['question'] else 'FAIL'}")
    print(f"  Task 2: {'PASS' if 'step_by_step_trace' in t2_sample['question'] else 'FAIL'}")

    # ── Self-correction balance ───────────────────────────────────────────
    sc_items  = [x for x in all_items if x["category"] == "self_correction"]
    wrong_pct = sum(1 for x in sc_items
                    if not x["ground_truth_answer"]["student_correct"]) / len(sc_items) * 100
    print(f"\n─── Self-correction wrong rate: {wrong_pct:.1f}% (target ~50%) ───")

    print("\nDone.")

Generating v7.0 Dataset (200 items × 6 categories = 1200 total)...

Task 1 saved : 1,000 items → metacog_task1_graph_tracing.jsonl
Task 2 saved : 200 items → metacog_task2_self_correction.jsonl
Total        : 1,200 items

─── Answer diversity ───
  [              easy]  total=200  solvable=200  unsolvable=  0  unique_answers= 10
  [              hard]  total=200  solvable=200  unsolvable=  0  unique_answers= 17
  [   novel_synthetic]  total=200  solvable=200  unsolvable=  0  unique_answers= 16
  [         ambiguous]  total=200  solvable=  0  unsolvable=200  unique_answers=  0
  [        impossible]  total=200  solvable=  0  unsolvable=200  unique_answers=  0
  [   self_correction]  total=200  solvable=200  unsolvable=  0  unique_answers= 12

─── Impossible type distribution (target: equal thirds) ───
  div_zero            :  67  (expected 66)
  circular            :  67  (expected 66)
  phantom             :  66  (expected 66)

─── Impossible correctness check ───
  All impossible item